# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PalSoham/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring (Lane 2)  
**Iteration partition:** `month=2026-03` (mid-panel month — safe for feature and label development)  
**Sealed test partition:** `month=2026-06` (never touched until final evaluation)

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Token safety:** if running in Colab, store your Hugging Face READ token as a Colab Secret named `HF_TOKEN` — never paste it into a cell. This repo is public.

In [1]:
# ── Setup: load credentials and connect to warehouse ──────────────────────
import os, pandas as pd, numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    RUNTIME = 'colab'
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    RUNTIME = 'local'

print(f'Runtime: {RUNTIME}')
print(f'HF token present: {bool(HF_TOKEN)}')

# ── Try warehouse (DuckDB over Parquet on HF) ─────────────────────────────
USE_WAREHOUSE = False
if HF_TOKEN:
    try:
        import duckdb
        con = duckdb.connect()
        con.execute("INSTALL httpfs; LOAD httpfs;")
        con.execute(f"SET s3_endpoint='huggingface.co'; SET s3_access_key_id='user'; "
                    f"SET s3_secret_access_key='{HF_TOKEN}';")
        HF_BASE = "hf://datasets/FlyRank/internship-warehouse"
        USE_WAREHOUSE = True
        print('Warehouse connection: OK')
    except Exception as e:
        print(f'Warehouse not available: {e}')

# ── Fallback: starter CSV (always available in repo) ──────────────────────
for p in ['../../data/raw/content_refresh_anonymized.csv',
          'data/raw/content_refresh_anonymized.csv']:
    if os.path.exists(p):
        CSV_PATH = p
        break
df_starter = pd.read_csv(CSV_PATH)
print(f'Starter CSV loaded: {df_starter.shape}')


Runtime: local
HF token present: False
Starter CSV loaded: (30000, 44)


## 1. Unit of analysis + time window

### The five contract answers

| # | Question | Answer |
|---|---|---|
| 1 | **One row means** | one pseudonymized content page, with aggregated signals measured over a trailing 90-day window |
| 2 | **Table(s) used** | starter: `content_refresh_anonymized.csv`; warehouse: `dim_content` joined to `fact_content_daily_performance` (partitioned by `month`) |
| 3 | **Time window** | feature window = trailing 90 days before the decision point; development partition = `month=2026-03` (mid-panel, safe); sealed test = `month=2026-06` (never touched for development) |
| 4 | **What I predict or rank** | proxy label — `is_declining_label = 1` when `trend_direction == 'down'` (impressions last 30d dropped >20% vs prior 30d); for capstone: a forward-looking label (impressions in next-30d < 80% of prior-30d, defined strictly after the feature window) |
| 5 | **One deliberate exclusion** | `trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d` — these are the label source columns; using them as features is the canonical leakage trap for this lane |

### Time-window diagram

```
                    feature window                  | label window
  |<-- 90-day aggregated signals (impressions,    -->| last 30d vs prev 30d
  |    clicks, sessions, position, CTR, age...)     | (NEVER a feature)
  |                                                  |
  day -90                                         day 0  (decision point)
```

For the warehouse forward-looking label (capstone):

```
  |<-- feature window (90d) -->|<-- gap -->|<-- label window (30d) -->|
  day -120                   day -30     day 0                      day +30
```

In [2]:
# ── Query 1: Grain verification ─────────────────────────────────────────
# Claim: one row = one content page (content_id is the grain key)

if USE_WAREHOUSE:
    q_grain = """
        SELECT content_hash_id, COUNT(*) AS c
        FROM parquet_scan('{HF_BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY content_hash_id
        HAVING c > 1
        LIMIT 5
    """.format(HF_BASE=HF_BASE)
    grain_check = con.execute(q_grain).df()
    print('Grain violation rows (expect 0 for starter, >0 for daily fact table):')
    print(grain_check)
    print('Note: fact_content_daily_performance grain is report_date x client x content')
    print('So many rows per content_id is EXPECTED for the daily table.')
    print('The correct grain check for this table is the composite key.')
else:
    # Starter CSV: one row per content page
    dupes = df_starter[df_starter.duplicated(subset='content_id', keep=False)]
    print('=== Grain check: starter CSV ===')
    print(f'Total rows: {len(df_starter)}')
    print(f'Duplicate content_id rows: {len(dupes)}  (expect 0 — one row per page)')
    print(f'Unique content_id: {df_starter["content_id"].nunique()}')
    print('GRAIN HOLDS: one row = one content page in the starter dataset')


=== Grain check: starter CSV ===
Total rows: 30000
Duplicate content_id rows: 0  (expect 0 — one row per page)
Unique content_id: 30000
GRAIN HOLDS: one row = one content page in the starter dataset


## 2. Fields: feature / label / context / excluded

### Features (safe to use — all knowable before the decision moment)

| Field | Type | Why available at decision time |
|---|---|---|
| `impressions_90d` | numeric | aggregated from GSC before the decision point |
| `clicks_90d` | numeric | aggregated from GSC before the decision point |
| `sessions_90d` | numeric | aggregated from GA4 before the decision point |
| `avg_position` | numeric | mean GSC rank over the 90d window (0 = no data, not rank zero) |
| `ctr` | numeric (×100%) | clicks/impressions ratio, 90d window |
| `engagement_rate` | numeric (×100%) | engaged_sessions/sessions, 90d window |
| `scroll_rate` | numeric (×100%) | scroll_events/pageviews (can exceed 100 — see data dict) |
| `content_age_days` | numeric | days since page creation — known at any point in time |
| `days_since_last_update` | numeric | days since last edit — known at decision point |
| `word_count` | numeric | static content property, known at decision point |
| `days_with_impressions` | numeric | count of active impression days in 90d window |
| `days_with_sessions` | numeric | count of active session days in 90d window |
| `content_type` | categorical | `keyword article` / `feedly article` / `comparison article` |
| `position_tier` | categorical | bucket of avg_position — transparent, threshold-based |
| `impression_tier` | categorical | bucket of impressions_90d — transparent, threshold-based |
| `freshness_tier` | categorical | bucket of days_since_last_update — transparent, threshold-based |

### Label / proxy (the target — never a feature)

| Field | Role | Note |
|---|---|---|
| `trend_direction` | **label source** | derived from 30d window comparison — must be excluded from features |
| `trend_pct` | **label source** | the raw percentage used to compute trend_direction — must be excluded |
| `is_declining_label` | **target** | `1` when `trend_direction == 'down'`, else `0` — the thing the model predicts |

### Context (join / group / split only — never model inputs)

| Field | Use |
|---|---|
| `content_id` | unique page identifier — used for deduplication and row identity |
| `client_id` | used for **grouped train/test splits** (client-holdout) and per-client checks |

### Excluded (with reason)

| Field | Reason for exclusion |
|---|---|
| `impressions_last_30d` | **label source** — directly computes `trend_direction` |
| `impressions_prev_30d` | **label source** — directly computes `trend_direction` |
| `clicks_last_30d`, `sessions_last_30d`, `clicks_prev_30d`, `sessions_prev_30d` | analogous window-comparison columns; using them leaks the exact window the label is derived from |
| `provider_used`, `model_used` | which LLM generated the content — product metadata, not a safe content signal for discovery |
| `ai_traffic_pct`, `ai_sessions_90d` | AI-session signal is present in only ~2% of rows; including it as a main feature introduces systematic missingness by content type |
| Any raw URL, domain, query, title, or client name | not in the dataset (already removed before release) — stated here for completeness |

In [3]:
# ── Query 2: Row count, date span, and data availability ─────────────────
# Claim: the mid-panel month 2026-03 has ~X rows, dates span the month,
# and filtering with ga4_data_available IS TRUE yields a clean GA4 subset.

if USE_WAREHOUSE:
    q_counts = """
        SELECT
            COUNT(*) AS total_rows,
            MIN(report_date) AS min_date,
            MAX(report_date) AS max_date,
            COUNT(DISTINCT content_hash_id) AS distinct_pages,
            COUNT(DISTINCT client_hash_id) AS distinct_clients,
            ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
                  / COUNT(*), 2) AS pct_ga4_available
        FROM parquet_scan('{HF_BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    """.format(HF_BASE=HF_BASE)
    print(con.execute(q_counts).df().to_string(index=False))
else:
    # Starter CSV — equivalent checks
    print('=== Query 2 equivalent: starter CSV counts and availability ===')
    print(f'Total rows: {len(df_starter):,}')
    print(f'Unique pages: {df_starter["content_id"].nunique():,}')
    print(f'Unique clients: {df_starter["client_id"].nunique()}')
    print()
    # In the starter CSV, availability is proxied by sessions_90d > 0 (GA4 data present)
    ga4_rows = (df_starter['sessions_90d'] > 0).sum()
    print(f'Rows with GA4 sessions > 0 (proxy for ga4_data_available IS TRUE): {ga4_rows:,}')
    print(f'  = {ga4_rows/len(df_starter)*100:.1f}% of rows')
    print()
    # Position availability
    pos_rows = (df_starter['avg_position'] > 0).sum()
    print(f'Rows with avg_position > 0 (GSC position data present): {pos_rows:,}')
    print(f'  = {pos_rows/len(df_starter)*100:.1f}% of rows')
    print()
    # Content type breakdown
    print('Content type distribution:')
    print(df_starter['content_type'].value_counts().to_string())


=== Query 2 equivalent: starter CSV counts and availability ===
Total rows: 30,000
Unique pages: 30,000
Unique clients: 32

Rows with GA4 sessions > 0 (proxy for ga4_data_available IS TRUE): 17,842
  = 59.5% of rows

Rows with avg_position > 0 (GSC position data present): 28,795
  = 96.0% of rows

Content type distribution:
content_type
keyword article       17532
feedly article        10500
comparison article     1968
Name: count, dtype: int64


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries, run on the mid-panel month (`month=2026-03` in warehouse, or equivalent checks on the starter CSV):

- **Query A** — grain check (already run in Section 1)
- **Query B** — row count, date span, client count, GA4 availability (Section 2)
- **Query C** — missingness by content type (the pattern the data dictionary warns about)

The third query is critical: the data dictionary states that `feedly article` rows have ~100% missing keyword data (`search_volume`, `competition`, `cpc`), while `keyword article` and `comparison article` rows have most keyword fields populated. A blind `fillna(0)` would silently encode content type into every keyword feature.

In [4]:
# ── Query 3: Missingness by content_type ─────────────────────────────────
# Claim: keyword-data missingness follows content_type, not random.

if USE_WAREHOUSE:
    q_missing = """
        SELECT
            content_type,
            COUNT(*) AS rows,
            ROUND(100.0 * AVG(CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END), 1) AS pct_missing_search_volume,
            ROUND(100.0 * AVG(CASE WHEN word_count IS NULL THEN 1 ELSE 0 END), 1) AS pct_missing_word_count,
            ROUND(100.0 * AVG(CASE WHEN avg_position = 0 THEN 1 ELSE 0 END), 1) AS pct_no_position
        FROM parquet_scan('{HF_BASE}/dim_content/*.parquet')
        GROUP BY content_type
        ORDER BY rows DESC
    """.format(HF_BASE=HF_BASE)
    print(con.execute(q_missing).df().to_string(index=False))
else:
    print('=== Query 3: Missingness by content_type (starter CSV) ===')
    miss = df_starter.groupby('content_type').agg(
        rows=('content_id', 'count'),
        pct_missing_search_volume=('search_volume', lambda x: round(100*x.isna().mean(), 1)),
        pct_missing_word_count=('word_count', lambda x: round(100*x.isna().mean(), 1)),
        pct_no_position=('avg_position', lambda x: round(100*(x==0).mean(), 1)),
    ).reset_index()
    print(miss.to_string(index=False))
    print()
    print('FINDING: feedly article rows have ~100% missing search_volume.')
    print('A blind fillna(0) on search_volume silently encodes content_type.')
    print('Fix: add a has_keyword_data flag instead of filling with 0.')


=== Query 3: Missingness by content_type (starter CSV) ===
        content_type   rows  pct_missing_search_volume  pct_missing_word_count  pct_no_position
     keyword article  17532                        0.0                    28.1              3.1
      feedly article  10500                      100.0                     0.0              5.8
  comparison article   1968                        0.0                    25.2              4.4

FINDING: feedly article rows have ~100% missing search_volume.
A blind fillna(0) on search_volume silently encodes content_type.
Fix: add a has_keyword_data flag instead of filling with 0.


### Five features and the leakage trap

Below: build a minimal feature frame from the starter data, annotate each feature with its 'available when?' justification, then deliberately add a leaky column and watch the score jump — then remove it.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.preprocessing import LabelEncoder
import warnings; warnings.filterwarnings('ignore')

# ── Build the five-feature frame ────────────────────────────────────────────
df = df_starter.copy()
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Filter: same as starter pipeline
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()

# Feature engineering
import numpy as np
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['has_keyword_data'] = df['search_volume'].notna().astype(int)
df['position_available'] = (df['avg_position'] > 0).astype(int)
df['ctr_safe'] = df['ctr'].fillna(0)
df['age_days'] = df['content_age_days']

FIVE_FEATURES = [
    'log_impressions_90d',   # search volume proxy — knowable before decision point
    'days_with_impressions', # consistency of search presence — knowable before decision point
    'avg_position',          # current ranking quality — knowable before decision point
    'ctr_safe',              # click efficiency — knowable before decision point
    'age_days',              # content age — knowable at any time
]

print('=== Five-feature frame: available-when? notes ===')
notes = [
    ('log_impressions_90d',   'log1p(impressions_90d) — trailing 90d GSC aggregate, fully available before decision'),
    ('days_with_impressions', 'days in 90d window with >=1 impression — trailing aggregate, available before decision'),
    ('avg_position',          'mean GSC position in 90d window — trailing aggregate, available before decision'),
    ('ctr_safe',              'clicks/impressions x100 in 90d — trailing aggregate; 0-filled for missing (no impressions)'),
    ('age_days',              'content_age_days — static property of the page, always known at decision time'),
]
for feat, note in notes:
    print(f'  {feat:<26}: {note}')

# Prepare arrays
X5 = df[FIVE_FEATURES].fillna(0).values
y  = df['is_declining_label'].values
groups = LabelEncoder().fit_transform(df['client_id'])

cv = GroupKFold(n_splits=5)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

scores_5 = cross_val_score(rf, X5, y, cv=cv, groups=groups,
                            scoring='average_precision')
print(f'\n=== Honest 5-feature model (client-holdout CV) ===')
print(f'Avg Precision: {scores_5.mean():.3f} (+/- {scores_5.std():.3f})')
print(f'Base rate: {y.mean():.3f}')


=== Five-feature frame: available-when? notes ===
  log_impressions_90d       : log1p(impressions_90d) — trailing 90d GSC aggregate, fully available before decision
  days_with_impressions     : days in 90d window with >=1 impression — trailing aggregate, available before decision
  avg_position              : mean GSC position in 90d window — trailing aggregate, available before decision
  ctr_safe                  : clicks/impressions x100 in 90d — trailing aggregate; 0-filled for missing (no impressions)
  age_days                  : content_age_days — static property of the page, always known at decision time

=== Honest 5-feature model (client-holdout CV) ===
Avg Precision: 0.541 (+/- 0.048)
Base rate: 0.542


### The deliberate leakage trap

Now add `trend_pct` — the raw percentage that directly defines the label — as a feature. Watch the score jump toward perfect. Then remove it and keep the honest number. This is the canonical Lane 2 leakage trap.

In [6]:
# ── DELIBERATE LEAKAGE: add trend_pct (label source) as a feature ─────────
# trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100
# is_declining_label = 1 when trend_pct < -20
# So trend_pct essentially IS the label — adding it is pure leakage.

leak_col = 'trend_pct'
df[leak_col] = df[leak_col].fillna(0)  # fill the 3,388 blanks with 0

LEAKY_FEATURES = FIVE_FEATURES + [leak_col]
X_leak = df[LEAKY_FEATURES].fillna(0).values

scores_leak = cross_val_score(rf, X_leak, y, cv=cv, groups=groups,
                               scoring='average_precision')
print('=== LEAKY model (trend_pct included) ===')
print(f'Avg Precision: {scores_leak.mean():.3f} (+/- {scores_leak.std():.3f})')
print(f'Jump from honest baseline: +{scores_leak.mean()-scores_5.mean():.3f}')
print()
print('>>> This jump is the CONFESSION. trend_pct encodes the label directly.')
print('>>> A real improvement of this magnitude would require a new dataset.')
print()
print('=== REMOVING leaky column — keeping only the honest result ===')
df.drop(columns=[leak_col], errors='ignore', inplace=True)
print(f'Clean 5-feature Avg Precision: {scores_5.mean():.3f}  <- this is the number to report')
print(f'Leaky Avg Precision: {scores_leak.mean():.3f}  <- inflated by label leakage, discarded')


=== LEAKY model (trend_pct included) ===
Avg Precision: 0.961 (+/- 0.012)
Jump from honest baseline: +0.420

>>> This jump is the CONFESSION. trend_pct encodes the label directly.
>>> A real improvement of this magnitude would require a new dataset.

=== REMOVING leaky column — keeping only the honest result ===
Clean 5-feature Avg Precision: 0.541  <- this is the number to report
Leaky Avg Precision: 0.961  <- inflated by label leakage, discarded


## 4. Data limits

**Named limitation of this slice:**

**1. The proxy label is not a future outcome.**  
The starter label (`trend_direction == 'down'`) is computed from the same 90-day window as the features — the last 30 days versus the prior 30 days. This means the label describes the *current* trajectory, not what will happen next. A page labelled 'declining' may stabilise or recover in the next 30 days without any editorial action. The label cannot tell us whether a refresh would help; it only tells us the page is currently losing impressions.

**2. Unbalanced client history — the panel is not uniform.**  
Different clients have different amounts of history. In the warehouse, some clients have 17 months of daily data; others have 3. In the starter CSV, all rows represent a single 90-day snapshot. Using a global calendar window for feature aggregation would treat 'no data yet' as 'no traffic' for newer clients — a systematic error. Always check `dim_clients.gsc_data_start` before defining any time window on the full warehouse.

**3. GA4 data is absent for early rows — zeros are not silence.**  
In the warehouse daily fact table, rows before a client's `ga4_data_start` have GA4 columns zero-filled with `ga4_data_available = FALSE`. These zeros are not 'no engagement' — they mean 'tracking not yet active'. Always filter with `ga4_data_available IS TRUE` (not `!= FALSE`, which silently passes NULL rows). In this starter CSV, ~40% of rows have `sessions_90d = 0`, some of which are genuine zero-traffic pages and some of which are pre-tracking rows.

**4. Precision@50 on the starter slice may not generalise.**  
The starter dataset contains 30,000 rows from 32 clients. The full warehouse has 519,606 content items from 104 clients. Client-level heterogeneity in the full data may reduce the model's transferability. The 3× Precision@50 lift observed in the starter pipeline must be re-earned on the warehouse with time-aware validation before it can be claimed.

In [7]:
# ── Quantify the data limits named above ───────────────────────────────────

print('=== Limit 1: proxy label coverage ===')
print(f'Rows with trend_pct missing (impressions_prev_30d == 0): '
      f'{df_starter["trend_pct"].isna().sum()}')
print('These rows cannot have a trend label — treated as non-declining by default.')
print()

print('=== Limit 2: GA4 data availability ===')
zero_sessions = (df_starter['sessions_90d'] == 0).sum()
print(f'Rows with sessions_90d == 0: {zero_sessions} ({zero_sessions/len(df_starter)*100:.1f}%)')
print('Some of these are true zero-traffic pages; some are pre-GA4-tracking rows.')
print('Cannot distinguish without ga4_data_available flag (only in warehouse).')
print()

print('=== Limit 3: position data coverage ===')
no_pos = (df_starter['avg_position'] == 0).sum()
print(f'Rows with avg_position == 0 (no GSC position data): {no_pos} ({no_pos/len(df_starter)*100:.1f}%)')
print('Per data dictionary: 0 means no data, not position zero.')
print('These rows should not be used in position-tier comparisons.')
print()

print('=== Limit 4: client count ===')
print(f'Starter: {df_starter["client_id"].nunique()} clients')
print('Warehouse dim_clients: 104 clients')
print('Results from 32-client starter may not generalise to full 104-client warehouse.')


=== Limit 1: proxy label coverage ===
Rows with trend_pct missing (impressions_prev_30d == 0): 3388
These rows cannot have a trend label — treated as non-declining by default.

=== Limit 2: GA4 data availability ===
Rows with sessions_90d == 0: 12158 (40.5%)
Some of these are true zero-traffic pages; some are pre-GA4-tracking rows.
Cannot distinguish without ga4_data_available flag (only in warehouse).

=== Limit 3: position data coverage ===
Rows with avg_position == 0 (no GSC position data): 1205 (4.0%)
Per data dictionary: 0 means no data, not position zero.
These rows should not be used in position-tier comparisons.

=== Limit 4: client count ===
Starter: 32 clients
Warehouse dim_clients: 104 clients
Results from 32-client starter may not generalise to full 104-client warehouse.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Five contract answers stated in Section 1
- [x] Three verification queries shown (grain, counts/availability, missingness by type)
- [x] Five features built with available-when? note each
- [x] Deliberate leakage trap shown (trend_pct), score jumped, column removed, honest number kept
- [x] One named limitation stated (proxy label is not a future outcome)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.